### Silver – Transactions

#### Purpose
Transform the Bronze `transactions` table into a clean and analytics-ready
Silver table by:
- Standardizing column names using a reusable UDF
- Enforcing correct data types
- Handling nulls based on business rules
- Deduplicating records
- Isolating malformed records into a quarantine table

#### Source
- coffee.bronze.transactions

#### Targets
- coffee.silver.transactions
- coffee.silver.quarantine_transactions


In [0]:
%python
# Notebook Configuration using Databricks Widgets
# Purpose:
# - Avoid hard coding environment-specific values
# - Make the code reusable across tables and environments
dbutils.widgets.text("catalog", "coffee")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("source_table", "transactions")
dbutils.widgets.text("default_watermark", "1900-01-01")

# Read widget values into Python variables
# These variables are used throughout the notebook
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
source_table = dbutils.widgets.get("source_table")
default_watermark = dbutils.widgets.get("default_watermark")


In [0]:
%run ./Silver_utils/silver_transform_utils

In [0]:
%python
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{source_table}")


In [0]:
%python
# Standardize all incoming column names using the central UDF.
# This ensures consistent snake_case naming across all Silver tables,
# regardless of how the raw files were named in Bronze.
df_std = standardize_columns(df_bronze)


In [0]:
%python
df_std.createOrReplaceTempView(f"bronze_{source_table}_std")


In [0]:
%python
# creating silver table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table} (
  transaction_id STRING,
  store_id INT,
  payment_method_id INT,
  voucher_id INT,
  user_id INT,
  original_amount DOUBLE,
  discount_applied DOUBLE,
  final_amount DOUBLE,
  created_at TIMESTAMP,
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP,
  load_dt DATE,
  source STRING,
  source_table STRING,
  source_file STRING,
  silver_loaded_at TIMESTAMP,
  silver_updated_at TIMESTAMP
)
USING DELTA
""")


In [0]:
-- Incremental extraction:
-- Only process Bronze rows that arrived after the latest loaded_at timestamp
-- already present in the Silver target table.
--
-- This prevents reprocessing old Bronze records and keeps Silver rerun-safe.

CREATE OR REPLACE TEMP VIEW bronze_incremental AS
SELECT *
FROM bronze_transactions_std
WHERE loaded_at >
(
  SELECT COALESCE(MAX(loaded_at), '1900-01-01')
  FROM coffee.silver.transactions
);


In [0]:
%python
# ---------------------------------------------
#  Count invalid records for transactions
# ---------------------------------------------
# If invalid_count = 0, we skip quarantine table creation and merge.

invalid_count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM bronze_incremental
WHERE
  transaction_id IS NULL
  OR created_at IS NULL
  OR original_amount IS NULL
  OR final_amount IS NULL
""").collect()[0]["cnt"]

print("Invalid transaction rows:", invalid_count)


In [0]:
%python

# Step 2: Create and load quarantine table only if invalid rows exist


if invalid_count > 0:

 
    # 2A: Create transactions quarantine table

    spark.sql("""
    CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table}_quarantine  (
      transaction_id STRING,
      store_id STRING,
      payment_method_id STRING,
      voucher_id STRING,
      user_id STRING,
      original_amount STRING,
      discount_applied STRING,
      final_amount STRING,
      created_at STRING,

      -- Bronze metadata
      loaded_at TIMESTAMP,
      updated_at TIMESTAMP,
      load_dt DATE,
      source STRING,
      source_file STRING,

      -- Quarantine metadata
      quarantine_reason STRING,
      quarantined_at TIMESTAMP
    )
    USING DELTA
    """)

 
    #  Merge invalid rows into quarantine (idempotent)
 
    spark.sql("""
    MERGE INTO {catalog}.{silver_schema}.{source_table}_quarantine  q
    USING (
      SELECT
        *,
        CASE
          WHEN transaction_id IS NULL THEN 'transaction_id is null'
          WHEN created_at IS NULL THEN 'created_at is null'
          WHEN original_amount IS NULL THEN 'original_amount is null'
          WHEN final_amount IS NULL THEN 'final_amount is null'
          ELSE 'unknown validation failure'
        END AS quarantine_reason,
        current_timestamp() AS quarantined_at
      FROM bronze_incremental
      WHERE
        transaction_id IS NULL
        OR created_at IS NULL
        OR original_amount IS NULL
        OR final_amount IS NULL
    ) b
    ON q.transaction_id = b.transaction_id
    AND q.quarantine_reason = b.quarantine_reason
    WHEN NOT MATCHED THEN
    INSERT *;
    """)

else:
    print("No invalid transaction rows found. Quarantine table not created.")


In [0]:
%python
# MERGE Bronze Incremental Transactions into Silver
# Purpose:
#1. Insert new transactions into Silver
# 2. Update existing transactions with latest values
# 3. Ensure idempotency (safe re-runs)
spark.sql(f""" 
MERGE INTO  {catalog}.{silver_schema}.{source_table} s
USING (
  SELECT
    transaction_id,
    TRY_CAST(store_id AS INT) AS store_id,
    TRY_CAST(payment_method_id AS INT) AS payment_method_id,
    TRY_CAST(voucher_id AS INT) AS voucher_id,
    TRY_CAST(user_id AS INT) AS user_id,
    TRY_CAST(original_amount AS DOUBLE) AS original_amount,
    TRY_CAST(discount_applied AS DOUBLE) AS discount_applied,
    TRY_CAST(final_amount AS DOUBLE) AS final_amount,
    TRY_CAST(created_at AS TIMESTAMP) AS created_at,

    -- Bronze metadata (propagated for lineage & debugging)
    loaded_at,
    updated_at,
    load_dt,
    source,
    'coffee.bronze.transactions' AS source_table,
    source_file,
    current_timestamp() AS silver_updated_at
  FROM (
    -- Deduplicate records within the incremental batch
    -- Keep the latest version per transaction_id
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY transaction_id
             ORDER BY updated_at DESC
           ) AS rn
    FROM bronze_incremental
    -- Only valid records move to Silver
    -- Invalid ones are handled in quarantine separately

    WHERE
      transaction_id IS NOT NULL
      AND created_at IS NOT NULL
      AND original_amount IS NOT NULL
      AND final_amount IS NOT NULL
  )
  WHERE rn = 1
) b
ON s.transaction_id = b.transaction_id
-- CASE 1: Record already exists in Silver → UPDATE
WHEN MATCHED THEN
  UPDATE SET
    s.store_id = b.store_id,
    s.payment_method_id = b.payment_method_id,
    s.voucher_id = b.voucher_id,
    s.user_id = b.user_id,
    s.original_amount = b.original_amount,
    s.discount_applied = b.discount_applied,
    s.final_amount = b.final_amount,
    s.created_at = b.created_at,
    s.updated_at = b.updated_at,
    s.load_dt = b.load_dt,
    s.source = b.source,
    s.source_file = b.source_file,
    s.silver_updated_at = b.silver_updated_at
-- CASE 2: Record does not exist in Silver → INSERT
WHEN NOT MATCHED THEN
  INSERT (
    transaction_id,
    store_id,
    payment_method_id,
    voucher_id,
    user_id,
    original_amount,
    discount_applied,
    final_amount,
    created_at,
    loaded_at,
    updated_at,
    load_dt,
    source,
    source_table,
    source_file,
    silver_loaded_at,
    silver_updated_at
  )
  VALUES (
    b.transaction_id,
    b.store_id,
    b.payment_method_id,
    b.voucher_id,
    b.user_id,
    b.original_amount,
    b.discount_applied,
    b.final_amount,
    b.created_at,
    b.loaded_at,
    b.updated_at,
    b.load_dt,
    b.source,
    b.source_table,
    b.source_file,
    current_timestamp(),
    current_timestamp()
  )
  """)
